# Chapitre 2 — Data Wrangling

Nettoyage, imputation et feature engineering.

In [1]:
import os, sys, warnings, datetime
warnings.filterwarnings('ignore')
sys.path.insert(0, 'c:/Users/ahoudzi/apti/aptispace-datascience-projet/src')
import pandas as pd, numpy as np

RAW = 'c:/Users/ahoudzi/apti/aptispace-datascience-projet/data/raw/marketing_campaign.csv'
df_raw = pd.read_csv(RAW, sep='\t', engine='c')
print(f'Brut : {df_raw.shape}  | manquants : {df_raw.isnull().sum().sum()}')
df_raw.head(3)

Brut : (2240, 29)  | manquants : 24


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0


## Downcasting — optimisation memoire

In [2]:
mem_av = df_raw.memory_usage(deep=True).sum()/1024
for c in df_raw.select_dtypes('int64').columns:
    df_raw[c] = pd.to_numeric(df_raw[c], downcast='integer')
for c in df_raw.select_dtypes('float64').columns:
    df_raw[c] = pd.to_numeric(df_raw[c], downcast='float')
mem_ap = df_raw.memory_usage(deep=True).sum()/1024
print(f'Avant : {mem_av:.1f} KB  |  Apres : {mem_ap:.1f} KB  |  Gain : {(1-mem_ap/mem_av)*100:.0f}%')

Avant : 561.4 KB  |  Apres : 187.4 KB  |  Gain : 67%


## Dates et anciennete client

In [3]:
df_raw['Dt_Customer'] = pd.to_datetime(df_raw['Dt_Customer'], dayfirst=True)
df_raw['Customer_Days'] = (df_raw['Dt_Customer'].max() - df_raw['Dt_Customer']).dt.days
print(df_raw[['Dt_Customer','Customer_Days']].head(3))

  Dt_Customer  Customer_Days
0  2012-09-04            663
1  2014-03-08            113
2  2013-08-21            312


## Outliers, flags et harmonisation

In [4]:
df_raw.loc[df_raw['Year_Birth'] < 1920, 'Year_Birth'] = np.nan
df_raw['Income_Missing_Flag'] = df_raw['Income'].isnull().astype('int8')
df_raw['Marital_Status'] = df_raw['Marital_Status'].replace({'YOLO':'Other','Absurd':'Other','Alone':'Other'})
print('Marital_Status:', df_raw['Marital_Status'].value_counts().to_dict())

Marital_Status: {'Married': 864, 'Together': 580, 'Single': 480, 'Divorced': 232, 'Widow': 77, 'Other': 7}


## Imputation mediane par strate Education × Marital_Status

In [5]:
df_raw['Income_Strata'] = df_raw['Education'] + '_' + df_raw['Marital_Status']
df_raw['Income'] = df_raw.groupby('Income_Strata')['Income'].transform(lambda x: x.fillna(x.median()))
df_raw['Income'] = df_raw['Income'].fillna(df_raw['Income'].median())
print('Manquants restants Income :', df_raw['Income'].isnull().sum())

Manquants restants Income : 0


## Feature engineering

In [6]:
df_raw['Age'] = datetime.date.today().year - df_raw['Year_Birth'].fillna(1970).astype(int)
spend = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
df_raw['Total_Spent'] = df_raw[spend].sum(axis=1)
edu_map = {'Basic':0,'2n Cycle':1,'Graduation':2,'Master':3,'PhD':4}
df_raw['Education'] = df_raw['Education'].map(edu_map).fillna(2).astype('int8')
print(df_raw[['Age','Total_Spent','Education']].describe().round(1))

          Age  Total_Spent  Education
count  2240.0       2240.0     2240.0
mean     57.1        605.8        2.5
std      11.7        602.2        1.0
min      30.0          5.0        0.0
25%      49.0         68.8        2.0
50%      56.0        396.0        2.0
75%      67.0       1045.5        3.0
max      86.0       2525.0        4.0


## Sauvegarde parquet

In [7]:
out = 'c:/Users/ahoudzi/apti/aptispace-datascience-projet/data/processed/marketing_clean.parquet'
df_raw.dropna(subset=['Year_Birth']).to_parquet(out, index=False)
df_check = pd.read_parquet(out)
print(f'Lignes : {len(df_check)} | Colonnes : {df_check.shape[1]} | Manquants : {df_check.isnull().sum().sum()}')

Lignes : 2237 | Colonnes : 34 | Manquants : 0
